# DETR3D in Pure PyTorch: Faithful End-to-End Walkthrough

This notebook explains the package implementation that reproduces the official DETR3D ResNet-101 base result. It is an annotated walkthrough, not a second copy of the model.

**Reproduced result:** 34.58 mAP / 42.15 NDS on the 6,019-sample nuScenes validation split. The upstream authors report 34.7 mAP / 42.2 NDS.

The result is a successful single-seed reproduction. It is not a claim of bit-identical optimization or multi-seed equivalence with upstream DDP training.

## What “Pure PyTorch” Means

The upstream project builds DETR3D with MMDetection, MMDetection3D, and MMCV. This repository replaces those runtime dependencies with local PyTorch modules:

| Upstream facility | Local implementation |
|---|---|
| MMDetection ResNet and MMCV DCNv2 | TorchVision ResNet-101 and modulated `DeformConv2d` wrapper |
| MMDetection FPN and DETR head | Local FPN and `Detr3DHead` |
| MMCV transformer layers | PyTorch attention and local decoder/cross-attention |
| MMDetection losses and assigner | Local focal/L1 losses and SciPy Hungarian assignment |
| MMDetection3D nuScenes dataset/CBGS | Local dataset and deterministic CBGS wrapper |
| MMDetection3D evaluation wrapper | Local exporter calling the official nuScenes devkit |

The official FCOS3D checkpoint is used only as initialization weights translated into local parameter names.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import torch

from detr3d.data.nuscenes_dataset import NUSCENES_CLASSES, NuScenesDetr3DDataset
from detr3d.engine.diagnostics import decode_predictions
from detr3d.models.losses import Detr3DLoss
from detr3d.models.losses.loss_utils import decode_bbox_predictions, encode_bbox_targets
from eval import build_model

dataroot_text = os.environ.get("NUSCENES_DATAROOT")
checkpoint_text = os.environ.get("DETR3D_CHECKPOINT")
DATAROOT = Path(dataroot_text) if dataroot_text else None
CHECKPOINT = Path(checkpoint_text) if checkpoint_text else None
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"device={DEVICE}")
print(f"dataset configured={DATAROOT is not None}")
print(f"checkpoint configured={CHECKPOINT is not None}")

## 1. nuScenes Data Contract

Each sample contains six synchronized camera tensors, LiDAR-to-image matrices, semantic 9D ground-truth boxes, and class labels.

```text
images:          [6, 3, H, W]
lidar2img:       [6, 4, 4]
image_shape:     [6, 2]
gt_boxes_lidar: [N, 9] = [x, y, z, w, l, h, yaw, vx, vy]
gt_labels:       [N]
```

The faithful path uses the official train/validation scene split, official category mapping, LiDAR-frame velocity, range filtering, and zero-point annotation filtering.

In [ ]:
dataset = None
sample = None
if DATAROOT is not None:
    dataset = NuScenesDetr3DDataset(
        dataroot=str(DATAROOT),
        version="v1.0-trainval",
        image_size=(900, 1600),
        split="val",
        filter_gt_by_range=True,
        filter_zero_point_gt=True,
        official_image_preprocessing=True,
        official_gt_semantics=True,
    )
    sample = dataset[0]
    assert sample["images"].shape[0] == 6
    assert sample["img_metas"]["lidar2img"].shape == (6, 4, 4)
    assert sample["gt_boxes_lidar"].shape[-1] == 9
    print(f"validation samples={len(dataset)}")
    print(f"images={tuple(sample['images'].shape)}")
    print(f"ground truth={tuple(sample['gt_boxes_lidar'].shape)}")
else:
    print("Set NUSCENES_DATAROOT to execute dataset-dependent cells.")

## 2. LiDAR-to-Image Geometry

Camera and LiDAR captures have separate calibrated sensors and ego poses. The projection is genuinely LiDAR-to-image:

```text
LiDAR sensor
  -> LiDAR ego frame
  -> global frame
  -> camera ego frame
  -> camera sensor
  -> image pixels
```

For camera `c`, the dataset composes:

```text
lidar2img[c] = K[c] @ global_to_camera[c] @ lidar_to_global
```

Using native 900×1600 images avoids an intrinsic-scaling mismatch. Official preprocessing pads the height to 928 after geometry is defined for the 900×1600 image.

In [ ]:
if sample is not None:
    matrices = sample["img_metas"]["lidar2img"]
    shapes = sample["img_metas"]["image_shape"]
    assert torch.isfinite(matrices).all()
    print(f"projection matrices={tuple(matrices.shape)}")
    print(f"model image shapes={shapes.tolist()}")
else:
    print("Dataset-dependent geometry probe skipped.")

## 3. Official Image Preprocessing

The official ResNet path differs from standard ImageNet preprocessing:

1. Resize to 900×1600.
2. Apply multiview photometric distortion during training only.
3. Convert RGB to BGR.
4. Subtract `[103.530, 116.280, 123.675]` without standard-deviation scaling.
5. Pad to a multiple of 32, yielding 928×1600 tensors.

Validation disables photometric distortion. GridMask is a model-side training augmentation and is also disabled automatically in evaluation mode.

## 4. Faithful Model Path

```text
six cameras
  -> Caffe-style ResNet-101 with modulated DCNv2
  -> four-level FPN
  -> 900 learned object queries and initial 3D references
  -> six decoder layers
       self-attention
       multiview 3D-to-2D cross-attention
       FFN
       classification/regression
       detached reference refinement
  -> six class predictions and encoded 10D box predictions
```

Cross-attention projects each normalized 3D reference into every camera and feature level, samples image features, applies learned sigmoid camera/level weights, masks invalid projections, and sums the valid evidence.

In [ ]:
model = build_model(
    num_queries=900,
    backbone_name="resnet101",
    pretrained_backbone=False,
    official_image_backbone=True,
    use_grid_mask=True,
)

assert len(model.transformer.layers) == 6
assert model.transformer.num_queries == 900
assert model.head.box_dim == 10
assert len(model.head.cls_branches) == 6
assert len(model.head.reg_branches) == 6

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"parameters={parameter_count:,}")

## 5. Iterative Reference Refinement

The production path is `Detr3DModel.forward`, not the transformer module in isolation.

1. Split learned query embeddings into query state and query position.
2. Predict one initial normalized 3D reference per query from `query_pos`.
3. Use the current references to sample multiview features.
4. Predict class logits and encoded box regression at the current layer.
5. Refine normalized XY and Z from the regression branch.
6. Detach refined references before the next decoder layer.

Detaching matches upstream behavior: later layers receive refined geometry without backpropagating through the previous refinement update.

In [ ]:
query, query_pos = model.transformer.init_decoder_state(batch_size=1, device=torch.device("cpu"))
references = model.head.init_reference_points(query_pos)

assert query.shape == (1, 900, 256)
assert query_pos.shape == (1, 900, 256)
assert references.shape == (1, 900, 3)
assert ((references > 0) & (references < 1)).all()

print(f"query={tuple(query.shape)}")
print(f"initial references={tuple(references.shape)}")

## 6. Semantic 9D Boxes and Encoded 10D Training Boxes

Dataset boxes are readable semantic 9D values:

```text
[x, y, z, w, l, h, yaw, vx, vy]
```

Matching and regression use the official encoded 10D representation:

```text
[x, y, log(w), log(l), z, log(h), sin(yaw), cos(yaw), vx, vy]
```

Predictions are decoded back to semantic 9D boxes for diagnostics and nuScenes export.

In [ ]:
semantic_box = torch.tensor(
    [[1.0, 2.0, 0.5, 1.8, 4.2, 1.6, 0.3, 2.0, -0.5]],
    dtype=torch.float32,
)
encoded_box = encode_bbox_targets(semantic_box)
decoded_box = decode_bbox_predictions(encoded_box)

assert encoded_box.shape == (1, 10)
assert decoded_box.shape == (1, 9)
torch.testing.assert_close(decoded_box, semantic_box)

print(f"semantic shape={tuple(semantic_box.shape)}")
print(f"encoded shape={tuple(encoded_box.shape)}")

## 7. Hungarian Matching and Losses

The faithful objective uses:

| Setting | Value |
|---|---:|
| Matcher classification weight | 2.0 |
| Matcher box weight | 0.25 |
| Focal alpha / gamma | 0.25 / 2.0 |
| Classification loss weight | 2.0 |
| Box loss weight | 0.25 |
| Background normalization weight | 0.0 |
| Velocity regression weights | 0.2 / 0.2 |

Hungarian box cost uses encoded dimensions `0:8`, excluding velocity. Regression still supervises velocity at reduced weight. Auxiliary losses supervise every decoder layer.

In [ ]:
criterion = Detr3DLoss(num_classes=len(NUSCENES_CLASSES))

assert criterion.matcher.cls_weight == 2.0
assert criterion.matcher.bbox_weight == 0.25
assert criterion.alpha == 0.25
assert criterion.gamma == 2.0
assert criterion.bg_cls_weight == 0.0
assert criterion.use_auxiliary_losses
torch.testing.assert_close(
    criterion.code_weights[-2:],
    torch.tensor([0.2, 0.2]),
)

print("Faithful matcher and loss settings verified.")

## 8. FCOS3D Initialization

The successful run did not start the image path from generic ImageNet weights. It translated 690 tensors from the official FCOS3D checkpoint, giving complete backbone/FPN coverage. The local loader maps upstream keys into the TorchVision/DCNv2/FPN modules.

Expected checkpoint location:

```text
checkpoints/fcos3d.pth
```

Checkpoint SHA-256 used for the reproduction:

```text
b6cd0590879adc21d2a54bebde44811391f006434a2ffcdc6b41480b2e95be48
```

## 9. Faithful 24-Epoch Training Recipe

| Setting | Value |
|---|---:|
| Train split | 28,130 samples |
| Physical batch | 4 |
| Accumulation | 2 |
| Effective batch | 8 |
| Queries | 900 |
| Image size | 900×1600 |
| Epochs | 24 |
| Optimizer | AdamW |
| Base/backbone LR | 2e-4 / 2e-5 |
| Weight decay | 0.01 |
| Warmup | 500 optimizer updates |
| Schedule | cosine to 1e-3 of initial LR |
| Gradient clipping | 35 |
| AMP | disabled |

The exact launch and recovery commands are maintained in `COMMAND_GUIDE.md` and `docs/analysis/c6-full-training.md`. They are intentionally not duplicated as executable notebook training cells because full training takes multiple days and generates large artifacts.

## 10. Optional Checkpoint Inference

Set `DETR3D_CHECKPOINT` and `NUSCENES_DATAROOT` to execute this cell. The checkpoint configuration is faithful C6, and rendering uses a display-only threshold of 0.3. Official metric evaluation uses flattened query-class top-300 decoding instead.

In [ ]:
if sample is not None and CHECKPOINT is not None:
    checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.to(DEVICE).eval()
    boxes, scores, labels = decode_predictions(
        model=model,
        sample=sample,
        device=DEVICE,
        score_threshold=0.3,
        max_boxes=100,
    )
    print(f"display predictions={len(scores)}")
    print(f"top scores={scores[:10].tolist()}")
else:
    print("Checkpoint inference skipped; configure both environment variables to run it.")

## 11. Diagnostics Versus Official Evaluation

The lightweight diagnostic computes nearest prediction centers and is useful for regression checks. It is **not** mAP/NDS because it can reuse one prediction for multiple GT boxes and does not fully penalize false positives.

Official evaluation performs:

1. Flattened query-class sigmoid scoring.
2. Global top-300 selection.
3. Post-center-range filtering.
4. LiDAR-to-global coordinate conversion.
5. Class-range filtering and nuScenes attribute assignment.
6. Direct invocation of the official nuScenes detection evaluator.

Use `eval.py --run-nuscenes-eval`; see `COMMAND_GUIDE.md` for the complete command.

## 12. Results

| Implementation | Variant | Split | mAP | NDS |
|---|---|---|---:|---:|
| Official DETR3D | ResNet-101 + DCN + GridMask | val | 34.7 | 42.2 |
| Official DETR3D | ResNet-101 + DCN + GridMask + CBGS | val | 34.9 | 43.4 |
| Official DETR3D | VoVNet + GridMask + CBGS | test | 41.2 | 47.9 |
| Pure PyTorch, earlier non-faithful run | Historical local configuration | val | 31.73 | 38.75 |
| **Pure PyTorch, faithful C6 run** | **ResNet-101 + DCN + GridMask** | **val** | **34.58** | **42.15** |

The faithful run improves the earlier implementation by 2.85 mAP points and 3.40 NDS points. It matches the upstream R101 base within 0.12 mAP points and 0.05 NDS points.

True-positive errors for the faithful run:

```text
mATE 0.7687   mASE 0.2711   mAOE 0.4007   mAVE 0.8664   mAAE 0.2070
```

The VoVNet row is not directly comparable because it uses another backbone, CBGS, trainval training, and test-set evaluation.

## 13. Verification Workflow

For implementation changes:

1. Run the unit suite.
2. Run the canonical deterministic one-sample regression.
3. Compare with the baseline recorded in `AGENTS.md` and `COMMAND_GUIDE.md`.
4. Run small controlled training before another full run.
5. Use official nuScenes mAP/NDS for promotion decisions.

Generated datasets, checkpoints, outputs, logs, MLflow stores, and notebook outputs are not committed to the repository.